# Thai Voice Translator — Colab Server  🇹🇭
รัน backend ทั้งหมดบน Colab GPU T4 ฟรี + ngrok tunnel

### วิธีใช้:
1. เปิด [Colab](https://colab.research.google.com/) → File → Open notebook → GitHub → ใส่ `ponsatornsam/thai-voice-translator`
2. หรือ: File → Upload → เลือกไฟล์นี้
3. **Runtime → Change runtime type → T4 GPU**
4. ใส่ [ngrok authtoken](https://dashboard.ngrok.com/get-started/your-authtoken) ใน Cell 6 (สมัครฟรี)
5. รันทีละ cell จากบนลงล่าง
6. เอา URL ที่ได้ไปใส่ใน Android app

> ⚠️ Colab session หมดอายุหลัง ~6-12 ชม. — ต้องรันใหม่ทุกครั้งที่หลุด

### Cell 1: ติดตั้ง dependencies (~2 นาที)

In [ ]:
!pip install -q fastapi uvicorn faster-whisper piper-tts httpx python-jose[cryptography] numpy soundfile pyngrok websockets
print("✅ Dependencies installed")

In [ ]:
# ติดตั้ง ffmpeg — ใช้สำหรับ Google TTS fallback
!apt install -y -qq ffmpeg 2>&1 | tail -1
print("✅ ffmpeg installed")

### Cell 2: ตั้งค่า API keys

In [ ]:
import os

os.environ["BACKEND_API_KEY"] = "your-backend-api-key-here"
os.environ["TRANSLATE_API_KEY"] = "your-gemini-api-key-here"
os.environ["TRANSLATE_API_BASE"] = "https://generativelanguage.googleapis.com/v1beta/openai"
os.environ["TRANSLATE_MODEL"] = "gemini-2.0-flash"
os.environ["PIPER_MODEL_DIR"] = "/content/models"
os.environ["PIPER_MODEL_NAME"] = "th_TH"

print("🔑 Keys configured")

### Cell 3: ดาวน์โหลด Piper Thai voice model (~1 นาที)

In [ ]:
import os
os.makedirs("/content/models", exist_ok=True)

# Piper binary
!wget -q https://github.com/rhasspy/piper/releases/download/v1.2.0/piper_linux_x86_64.tar.gz -O /tmp/piper.tar.gz
!tar -xzf /tmp/piper.tar.gz -C /tmp/
!cp /tmp/piper/piper /usr/local/bin/piper
!chmod +x /usr/local/bin/piper
!rm -rf /tmp/piper /tmp/piper.tar.gz

# Thai voice model (medium quality)
!wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/th/th_TH/vits-th/th_TH-medium.onnx -O /content/models/th_TH.onnx
!wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/th/th_TH/vits-th/th_TH-medium.onnx.json -O /content/models/th_TH.json

!ls -lh /content/models/
!echo "Piper binary:" && /usr/local/bin/piper --help 2>&1 | head -1
print("\n✅ Piper model + binary ready")

### Cell 4: โหลด Whisper model บน GPU (~30 วิ)

In [ ]:
from faster_whisper import WhisperModel
import torch

if torch.cuda.is_available():
    print(f"🟢 GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    device, compute = "cuda", "int8"
else:
    print("🔴 No GPU! Runtime → Change runtime type → T4 GPU")
    device, compute = "cpu", "int8"

model = WhisperModel("base", device=device, compute_type=compute)
print(f"✅ Whisper model loaded on {device.upper()}")

### Cell 5: เริ่ม FastAPI server

In [ ]:
import sys, threading, time
import uvicorn

# Clone backend code
!rm -rf /content/repo /content/*.py
!git clone -q https://github.com/ponsatornsam/thai-voice-translator.git /content/repo
!cp /content/repo/backend/*.py /content/

sys.path.insert(0, "/content")

# Patch Piper model path for Colab
import tts_service
tts_service.MODEL_DIR = "/content/models"
tts_service.MODEL_NAME = "th_TH"
tts_service.MODEL_PATH = "/content/models/th_TH.onnx"
tts_service.MODEL_CONFIG = "/content/models/th_TH.json"
tts_service._model_available = True

# Patch: update translate API key from env (in case Cell 2 was re-run)
import translate_service
translate_service.API_KEY = os.environ.get("TRANSLATE_API_KEY", "")
translate_service.API_BASE = os.environ.get("TRANSLATE_API_BASE", translate_service.API_BASE)
# Re-apply Gemini auto-correction
if "generativelanguage.googleapis.com" in translate_service.API_BASE and "/openai" not in translate_service.API_BASE:
    translate_service.API_BASE = translate_service.API_BASE.rstrip("/") + "/openai"
print(f"🔑 Translate API: key={'***' if translate_service.API_KEY else 'MISSING!'} · base={translate_service.API_BASE}")

# Patch: inject preloaded Whisper model
import whisper_service
whisper_service._model = model

def run_server():
    uvicorn.run("main:app", host="0.0.0.0", port=8000, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)
print("✅ FastAPI server running on port 8000")

### Cell 6: สร้าง ngrok tunnel → ได้ URL สาธารณะ
**ต้องใส่ ngrok authtoken ก่อน** — ขอฟรีที่ https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
from pyngrok import ngrok

# ⚠️ ใส่ authtoken ของคุณตรงนี้ (สมัครฟรีที่ ngrok.com)
NGROK_AUTHTOKEN = "your-ngrok-authtoken-here"

ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()

tunnel = ngrok.connect(8000, "http")
public_url = tunnel.public_url
ws_url = public_url.replace("https://", "wss://").replace("http://", "ws://")

api_key = os.environ["BACKEND_API_KEY"]

print("=" * 65)
print("🔗  Android WebSocket URL (เอาไปใส่ใน app):")
print(f"    {ws_url}/ws/audio?api_key={api_key}")
print("=" * 65)
print(f"💚 Health check:      {public_url}/health")
print(f"📊 Rate limits:       {public_url}/admin/rate-limits?api_key={api_key}")
print(f"🔍 Pipeline status:   {public_url}/admin/pipeline-status?api_key={api_key}")
print("=" * 65)
print("⚠️  Session หมดอายุใน 6-12 ชม. — ต้องรันใหม่ทุกครั้ง")
print("⚠️  URL เปลี่ยนทุกครั้งที่รัน — ต้องอัปเดตใน Android app ด้วย")

### Cell 7: ตรวจเช็คทุกอย่าง — Pipeline Diagnostics
เช็คทีละขั้น: Health → Auth → Pipeline Status → WebSocket → ASR → Translation → TTS

In [ ]:
import urllib.request, urllib.error, json, time, ssl, threading
from datetime import datetime

# ── Config ───────────────────────────────────────────────────────────
HEALTH_URL = f"{public_url}/health"
STATUS_URL = f"{public_url}/admin/pipeline-status?api_key={api_key}"
WS_URL = f"{ws_url}/ws/audio?api_key={api_key}"

PASS = "✅"
FAIL = "❌"
WARN = "⚠️"

def http_get(url, timeout=8):
    """HTTP GET — returns (status_code, body_str, latency_ms)."""
    ctx = ssl.create_default_context()
    req = urllib.request.Request(url, headers={"ngrok-skip-browser-warning": "true"})
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=timeout, context=ctx) as resp:
            body = resp.read().decode()
            latency = int((time.time() - t0) * 1000)
            return resp.status, body, latency
    except urllib.error.HTTPError as e:
        latency = int((time.time() - t0) * 1000)
        return e.code, e.read().decode()[:300], latency
    except Exception as e:
        latency = int((time.time() - t0) * 1000)
        return 0, str(e)[:200], latency

def check(label, ok, detail, latency=None):
    icon = PASS if ok else FAIL
    lat = f" · {latency}ms" if latency and latency > 0 else ""
    print(f"  {icon} {label:<28} {detail}{lat}")

all_ok = True

print("=" * 60)
print("🔍 Thai Voice Translator — Pipeline Diagnostics")
print(f"   {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   Server: {public_url}")
print("=" * 60)

# ── 1. Server Health ────────────────────────────────────────────────
status, body, lat = http_get(HEALTH_URL)
ok = status == 200 and "ok" in body
if not ok: all_ok = False
check("1. Server Health", ok, f"HTTP {status} · {body[:60]}", lat)

# ── 2. API Authentication ────────────────────────────────────────────
status, body, lat = http_get(f"{public_url}/admin/rate-limits?api_key={api_key}")
ok = status == 200
if not ok: all_ok = False
detail = f"HTTP {status}"
if ok:
    try:
        data = json.loads(body)
        auth = "on" if data.get("auth_enabled") else "off"
        limit = data.get("stats", {}).get("limit_per_ip", "?")
        detail = f"auth={auth} · limit={limit}/IP"
    except: pass
check("2. API Auth", ok, detail, lat)

# ── 3. Pipeline Status ───────────────────────────────────────────────
pipeline_status_code, body, lat = http_get(STATUS_URL)
ok = pipeline_status_code == 200
if ok:
    try:
        ps = json.loads(body)
        svc = ps.get("services", {})
        svc_all_ready = ps.get("all_ready", False)
        if not svc_all_ready: all_ok = False
        
        check("3. Pipeline Status", svc_all_ready, "all_ready" if svc_all_ready else "some not ready", lat)
        
        for name in ["asr", "translate", "tts", "ffmpeg"]:
            s = svc.get(name, {})
            ready = s.get("ready", False)
            detail = s.get("detail", "")
            method = s.get("method", "")
            if method: detail = f"[{method}] {detail}"
            check(f"   ↳ {name}", ready, detail[:80])
    except Exception as e:
        check("3. Pipeline Status", False, f"Parse error: {e}", lat)
        all_ok = False
else:
    check("3. Pipeline Status", False, f"HTTP {pipeline_status_code} — endpoint missing? (old code?)", lat)
    all_ok = False

# ── 4. WebSocket Handshake Test ──────────────────────────────────────
print(f"\n  ⏳ 4. WebSocket Handshake — testing...", end="", flush=True)

ws_result = {"ok": False, "lat": 0, "error": ""}

def _run_ws_test():
    """Run async WebSocket test in a dedicated thread (Colab-safe)."""
    import asyncio
    try:
        import websockets
    except ImportError:
        ws_result["error"] = "websockets not installed"
        return
    
    async def _connect():
        t0 = time.time()
        try:
            async with websockets.connect(
                WS_URL,
                extra_headers={"ngrok-skip-browser-warning": "true"},
                close_timeout=5,
                open_timeout=10,
            ) as wsock:
                ws_result["lat"] = int((time.time() - t0) * 1000)
                ws_result["ok"] = True
        except Exception as e:
            ws_result["lat"] = int((time.time() - t0) * 1000)
            ws_result["error"] = str(e)[:100]
    
    try:
        asyncio.run(_connect())
    except RuntimeError:
        # Fallback: use event loop in this thread
        try:
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            loop.run_until_complete(_connect())
            loop.close()
        except Exception as e:
            ws_result["error"] = str(e)[:100]

# Run WS test in separate thread to avoid Colab event-loop conflicts
t = threading.Thread(target=_run_ws_test, daemon=True)
t.start()
t.join(timeout=20)

print(f"\r", end="")
if ws_result["ok"]:
    check("4. WebSocket", True, f"Connected · handshake {ws_result['lat']}ms", ws_result["lat"])
else:
    err = ws_result["error"] if ws_result["error"] else "Failed"
    # Try simple TCP fallback
    if not ws_result["ok"] and not ws_result["error"]:
        import socket
        from urllib.parse import urlparse
        t0 = time.time()
        try:
            parsed = urlparse(public_url)
            sock = socket.create_connection((parsed.hostname, parsed.port or 443), timeout=10)
            sock.close()
            ws_result["lat"] = int((time.time() - t0) * 1000)
            check("4. WebSocket", True, f"TCP reachable · {ws_result['lat']}ms (WS lib unavailable)", ws_result["lat"])
        except Exception as e2:
            check("4. WebSocket", False, f"TCP: {str(e2)[:50]}", ws_result["lat"])
            all_ok = False
    else:
        check("4. WebSocket", False, err[:60], ws_result["lat"])
        all_ok = False

# ── Summary ──────────────────────────────────────────────────────────
print()
print("=" * 60)
if all_ok:
    print(f"{PASS} ALL CHECKS PASSED — server ready for Android connection")
else:
    print(f"{FAIL} SOME CHECKS FAILED — see details above")
print("=" * 60)

print()
print("🔗 Android WebSocket URL:")
print(f"   {WS_URL}")
print()

### 🎉 เสร็จ! เซิร์ฟเวอร์กำลังรัน
- Colab cell 6 จะค้างอยู่ — **อย่าปิด tab นี้**
- เปิด Android Studio → เอา URL จากด้านบนไปใส่ใน `wsClient.connect()`
- Build APK → ทดสอบพูด → ได้ยินเสียงไทย!

**คราวหน้ามาใหม่:** เปิด notebook นี้ → รัน cell 1→6 ใหม่ (URL จะเปลี่ยน)